### Transform Drivers Data

1. Read bronze_drivers table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (driverId → driver_id, dateOfBirth → date_of_birth)
4. Concatenate name.givenName and name.familyName to create a new column called driver_name and transform the value to Title Case
5. Remove duplicate records
6. Transform values of column nationality to Title Case
7. Write the transformed data to silver_drivers table

In [0]:
%run ../00-common/01.environment-configuration

In [0]:
bronze_table = F"{catalog_name}.{bronze_schema}.drivers"
silver_table = F"{catalog_name}.{silver_schema}.drivers"

In [0]:
drivers_df = spark.read.table(bronze_table)


In [0]:
display(drivers_df)

### Keep only the columns required for analytics (Drop url column)

In [0]:
from pyspark.sql import functions as F

In [0]:
drivers_selected_df = drivers_df.drop("url")



In [0]:
display(drivers_selected_df)

### Standardise column names using snake_case (driverId → driver_id, dateOfBirth → date_of_birth)

In [0]:
drivers_renamed_df = (
    drivers_selected_df
        .withColumnsRenamed({
            "driverId": "driver_id", 
            "dateOfBirth": "date_of_birth"})
)


In [0]:
display(drivers_renamed_df)

### Concatenate name.givenName and name.familyName to create a new column called driver_name and transform the value to Title Case

In [0]:
drivers_concatenated_df =(
    drivers_renamed_df
        .withColumn("driver_name",F.initcap(F.concat_ws(" ", F.col("name.givenname"), F.col("name.familyname"))))
        .drop("name")
)

In [0]:
display(drivers_concatenated_df)

### Remove duplicate records

In [0]:
drivers_distinct_df = drivers_concatenated_df.dropDuplicates(['driver_id'])


In [0]:
display(drivers_distinct_df)

### Transform values of column nationality to Title Case

In [0]:
drivers_final_df = (
    drivers_distinct_df
        .withColumn("nationality", F.initcap(F.col("nationality")))

)

In [0]:
(
    drivers_final_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(silver_table)
)        

In [0]:
display(spark.read.table(silver_table))